# Frequentist chirp‑mass inference for an 8 s BNS signal

A **self‑contained** demonstration of getting the "posterior" of a gravitational‑wave
parameter the *frequentist* way — from the likelihood surface via Wilks' theorem and the
Fisher/Laplace approximation — instead of MCMC. We infer the **chirp mass alone** of an
8 s binary‑neutron‑star (BNS) signal, using a leading‑order (0PN) stationary‑phase
waveform so the whole notebook runs in seconds with only `numpy`/`scipy` (+ optional
`jax` for an exact autodiff Hessian).

The five steps:

1. **Likelihood** — build the Gaussian‑noise log‑likelihood `ln L(Mc)`.
2. **Confidence interval (Wilks)** — `−2 Δln L ~ χ²₁` ⇒ likelihood‑ratio intervals.
3. **Fisher / Laplace** — the Gaussian posterior from the curvature at the MLE (exact via autodiff).
4. **Coverage study** — repeat over many noise realizations; do the X% intervals cover the truth X% of the time?
5. **Bayesian cross‑check** — a 1‑D flat‑prior posterior on a grid, overlaid on the frequentist result (Bernstein–von Mises).

All non‑mass parameters (amplitude, coalescence time and phase) are held fixed and the PSD
is assumed known — the minimal setting in which the statistics are transparent.

## Waveform model — 0PN stationary‑phase approximation

In the stationary‑phase approximation the frequency‑domain CBC waveform is

$$\tilde h(f)=\mathcal{A}\,f^{-7/6}\,e^{i\Psi(f;\boldsymbol\theta)},$$

and at leading (0PN) order the phase depends on the component masses only through the
chirp mass $\mathcal{M}$,

$$\Psi_{0\mathrm{PN}}(f;\mathcal{M})=\frac{3}{128}\left(\frac{\pi G\mathcal{M}f}{c^{3}}\right)^{-5/3},
\qquad \mathcal{M}=\frac{(m_1 m_2)^{3/5}}{(m_1+m_2)^{1/5}}.$$

We drop the constant/linear‑in‑$f$ pieces (coalescence time $t_c$, phase $\phi_c$, the
$-\pi/4$) since they are held fixed. Note the amplitude $\propto f^{-7/6}$ is
$\mathcal{M}$‑independent here, so **all** chirp‑mass information lives in the phase — which
is why $\mathcal{M}$ is measured so sharply.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2, norm

rng = np.random.default_rng(0)

# --- physical constants (SI) ---
G, c, MSUN = 6.67430e-11, 2.99792458e8, 1.98892e30
TSUN = G * MSUN / c**3            # one solar mass in seconds (~4.925e-6 s)

# --- data segment ---
fs, T = 2048.0, 8.0              # sample rate [Hz], segment length [s] -> 8 s BNS snippet
N     = int(fs * T)
df    = 1.0 / T                  # frequency resolution [Hz]
freqs = np.fft.rfftfreq(N, d=1.0 / fs)
f_low, f_high = 20.0, 512.0      # analysis band
band  = (freqs >= f_low) & (freqs <= f_high)
fb    = freqs[band]              # band-limited frequency grid

# --- source: a BNS-like chirp mass; everything else fixed/known ---
Mc_true = 1.219                  # [Msun]  (~1.4 + 1.4 Msun)
SNR0    = 30.0                   # fiducial optimal SNR

## Detector noise, waveform and inner product

A toy but realistically *shaped* one‑sided PSD (seismic wall $\sim f^{-4}$, a bucket, shot
noise $\sim f^{2}$). Its absolute scale is irrelevant — we rescale the amplitude
$\mathcal{A}$ so the injection has a chosen optimal SNR.

In [2]:
def psd(f):
    f0 = 150.0
    return 1e-46 * ((f0 / f)**4 + 2.0 + 2.0 * (f / f0)**2)
Sb = psd(fb)

def phase_0pn(f, mc):
    "0PN stationary-phase GW phase Psi(f; Mc)."
    return (3.0 / 128.0) * (np.pi * mc * TSUN * f)**(-5.0 / 3.0)

def waveform(mc, amp):
    "0PN SPA template h~(f) = amp * f^{-7/6} exp(i Psi) on the band."
    return amp * fb**(-7.0 / 6.0) * np.exp(1j * phase_0pn(fb, mc))

def inner(a, b):
    "Noise-weighted inner product 4 Re Sum a* b / S_n df."
    return 4.0 * df * np.sum(np.conj(a) * b / Sb).real

# fix the amplitude so the injection has optimal SNR = SNR0
_h_unit = waveform(Mc_true, 1.0)
amp0    = SNR0 / np.sqrt(inner(_h_unit, _h_unit))
h_true  = waveform(Mc_true, amp0)
print(f"optimal SNR = {np.sqrt(inner(h_true, h_true)):.3f}")

fig, ax = plt.subplots(figsize=(8, 4.3))
ax.loglog(fb, np.sqrt(Sb), label=r"ASD $\sqrt{S_n(f)}$")
ax.loglog(fb, 2 * fb * np.abs(h_true), label=r"$2f\,|\tilde h(f)|$ (characteristic strain)")
ax.set_xlabel("frequency [Hz]"); ax.set_ylabel(r"[Hz$^{-1/2}$]")
ax.grid(which="both", alpha=0.3); ax.legend()
ax.set_title("0PN SPA BNS signal vs. detector noise"); plt.show()

optimal SNR = 30.000


## Step 1 — the log‑likelihood

For stationary Gaussian noise of known PSD, the log‑likelihood of data $d$ given a template
$h(\mathcal{M})$ is (up to an $\mathcal{M}$‑independent constant)

$$\ln L(\mathcal{M}) = \langle d\,|\,h(\mathcal{M})\rangle - \tfrac12\langle h(\mathcal{M})\,|\,h(\mathcal{M})\rangle .$$

Here $\langle h|h\rangle$ is constant in $\mathcal{M}$ (amplitude is phase‑independent), so the
likelihood is driven entirely by the matched filter $\langle d|h(\mathcal{M})\rangle$.
We draw one fiducial data set $d = h(\mathcal{M}_\mathrm{true}) + n$.

In [3]:
def draw_noise(size):
    # colored Gaussian noise in the frequency domain. For the inner product
    # 4 df Sum /S_n, consistency (Var<n|h> = <h|h>) requires E|n(f)|^2 = S_n/(2 df):
    # sqrt(S_n/(4 df)) * (N(0,1)+iN(0,1)) has variance S_n/(4 df)*2 = S_n/(2 df).
    scale = np.sqrt(Sb / (4.0 * df))
    return scale * (rng.standard_normal(size) + 1j * rng.standard_normal(size))

d0 = h_true + draw_noise(fb.size)

def loglik(mc, d, amp=amp0):
    h = waveform(mc, amp)
    return inner(d, h) - 0.5 * inner(h, h)

print(f"ln L at truth = {loglik(Mc_true, d0):.3f}   (~ optimal SNR^2 / 2 = {SNR0**2/2:.1f})")

ln L at truth = 467.948   (~ optimal SNR^2 / 2 = 450.0)


## Step 2 — confidence interval from Wilks' theorem

The likelihood‑ratio statistic $w(\mathcal{M}) = -2\big[\ln L(\mathcal{M}) - \ln L(\hat{\mathcal{M}})\big]$
is asymptotically $\chi^2_1$ distributed (**Wilks 1938**). A $100(1-\alpha)\%$ confidence
interval is the set where $w \le \chi^2_1(1-\alpha)$ — i.e. $\{1.0, 2.71, 3.84\}$ for
$\{68, 90, 95\}\%$. We evaluate $\ln L$ on a fine grid (fast, since the template depends on
$\mathcal{M}$ only through a phase).

In [4]:
# analytic Fisher (derived in Step 3) just to size the grid
def fisher_analytic(mc, amp=amp0):
    h  = waveform(mc, amp)
    dpsi_dmc = -(5.0 / 3.0) * phase_0pn(fb, mc) / mc      # dPsi/dMc = -(5/3) Psi/Mc
    dh = 1j * dpsi_dmc * h                                # dh/dMc (amplitude is Mc-independent)
    return inner(dh, dh)
sig0 = 1.0 / np.sqrt(fisher_analytic(Mc_true))

# fine chirp-mass grid, wide enough to hold intervals down to low SNR later
mc_grid = np.linspace(Mc_true - 20 * sig0, Mc_true + 20 * sig0, 2500)

# log-likelihood on the grid for the fiducial data (vectorised matched filter)
Hgrid = amp0 * fb[None, :]**(-7/6) * np.exp(1j * phase_0pn(fb[None, :], mc_grid[:, None]))
hh    = 4 * df * np.sum(np.abs(Hgrid)**2 / Sb, axis=1).real     # constant in Mc
Amat  = 4 * df * Hgrid / Sb                                     # weights for <d|h>
del Hgrid

def loglik_grid(d):
    return (Amat @ np.conj(d)).real - 0.5 * hh

lnL   = loglik_grid(d0)
k_hat = int(lnL.argmax()); mc_hat = mc_grid[k_hat]
w     = 2.0 * (lnL.max() - lnL)               # -2 Delta lnL ~ chi^2_1

def wilks_interval(cl):
    inside = w <= chi2.ppf(cl, df=1)
    return mc_grid[inside].min(), mc_grid[inside].max()

print(f"MLE  Mc = {mc_hat:.6f}   (truth {Mc_true})")
for cl in (0.68, 0.90, 0.95):
    lo, hi = wilks_interval(cl)
    print(f"  {int(cl*100)}% CI: [{lo:.6f}, {hi:.6f}]   half-width = {(hi-lo)/2:.2e}")

fig, ax = plt.subplots(figsize=(8, 4.3))
ax.plot(mc_grid, w, color="C0")
for cl, ls in zip((0.68, 0.90, 0.95), (":", "--", "-.")):
    ax.axhline(chi2.ppf(cl, 1), ls=ls, color="gray", label=f"{int(cl*100)}%  (χ²₁)")
ax.axvline(Mc_true, color="crimson", lw=1.2, label="truth")
ax.axvline(mc_hat, color="C2", lw=1.2, ls="--", label="MLE")
ax.set_xlim(Mc_true - 6*sig0, Mc_true + 6*sig0); ax.set_ylim(0, 12)
ax.set_xlabel(r"$\mathcal{M}$ [$M_\odot$]"); ax.set_ylabel(r"$-2\,\Delta\ln L$")
ax.legend(); ax.set_title("Step 2 — likelihood-ratio (Wilks) intervals"); plt.show()

MLE  Mc = 1.219027   (truth 1.219)
  68% CI: [1.219010, 1.219045]   half-width = 1.74e-05
  90% CI: [1.218998, 1.219056]   half-width = 2.88e-05
  95% CI: [1.218993, 1.219061]   half-width = 3.44e-05


## Step 3 — Fisher / Laplace posterior (exact Hessian via autodiff)

Near the maximum the log‑likelihood is approximately quadratic, so the posterior is Gaussian
with variance $\sigma_\mathcal{M}^2 = 1/F$, where the Fisher information
$F = \langle \partial_\mathcal{M} h\,|\,\partial_\mathcal{M} h\rangle$ equals
$-\,\partial^2_\mathcal{M}\ln L$ at the MLE in the high‑SNR limit. Because the waveform is
**differentiable**, the Hessian is exact from autodiff — no finite‑difference step‑size
tuning (the classic Fisher‑matrix pitfall). We compare three routes: analytic (expected
Fisher), grid curvature (observed), and JAX autodiff (observed).

In [5]:
F_analytic = fisher_analytic(Mc_true)                     # expected Fisher
sig_fisher = 1.0 / np.sqrt(F_analytic)

d2      = np.gradient(np.gradient(lnL, mc_grid), mc_grid)  # observed curvature
F_grid  = -d2[k_hat]

F_auto = np.nan
try:
    import jax, jax.numpy as jnp
    jax.config.update("jax_enable_x64", True)
    fbj, Sbj, d0j = jnp.asarray(fb), jnp.asarray(Sb), jnp.asarray(d0)
    def loglik_j(mc):
        psi = (3/128) * (jnp.pi * mc * TSUN * fbj)**(-5/3)
        h   = amp0 * fbj**(-7/6) * jnp.exp(1j * psi)
        dh  = (4*df * jnp.sum(jnp.conj(d0j) * h / Sbj)).real
        hh_ = 4*df * jnp.sum(jnp.abs(h)**2 / Sbj)
        return dh - 0.5 * hh_
    F_auto = float(-jax.hessian(loglik_j)(mc_hat))         # exact observed Hessian
except Exception as e:
    print("JAX unavailable, skipping autodiff Hessian:", e)

print(f"sigma_Mc  analytic  (expected Fisher) : {sig_fisher:.3e}")
print(f"sigma_Mc  grid      (observed)        : {1/np.sqrt(F_grid):.3e}")
print(f"sigma_Mc  autodiff  (observed)        : {1/np.sqrt(F_auto):.3e}")
print(f"fractional precision sigma_Mc/Mc      : {sig_fisher/Mc_true:.2e}")

like  = np.exp(lnL - lnL.max())
gauss = np.exp(-0.5 * (mc_grid - mc_hat)**2 / sig_fisher**2)
fig, ax = plt.subplots(figsize=(8, 4.3))
ax.plot(mc_grid, like, label="likelihood (exact)")
ax.plot(mc_grid, gauss, "--", label="Fisher / Laplace Gaussian")
ax.axvline(Mc_true, color="crimson", lw=1.2, label="truth")
ax.set_xlim(Mc_true - 6*sig_fisher, Mc_true + 6*sig_fisher)
ax.set_xlabel(r"$\mathcal{M}$ [$M_\odot$]"); ax.set_ylabel("normalised")
ax.legend(); ax.set_title("Step 3 — Fisher / Laplace posterior"); plt.show()

sigma_Mc  analytic  (expected Fisher) : 1.590e-05
sigma_Mc  grid      (observed)        : 1.757e-05
sigma_Mc  autodiff  (observed)        : 1.757e-05
fractional precision sigma_Mc/Mc      : 1.30e-05


## Step 4 — coverage study (the real frequentist test)

A confidence interval's claim is about **coverage**: over repeated noise realizations, do the
$X\%$ intervals contain the truth $X\%$ of the time? We regenerate the data over many noise
seeds (fixed injection), rebuild the Wilks intervals, and check. Two diagnostics:

* a **P–P plot** — for each realization, the confidence level at which the truth enters,
  $p = F_{\chi^2_1}\!\big(w(\mathcal{M}_\mathrm{true})\big)$, should be **uniform** ⇒ the
  empirical‑vs‑nominal curve lies on the diagonal;
* the **pull** $(\hat{\mathcal{M}}-\mathcal{M}_\mathrm{true})/\sigma$ should be $\mathcal{N}(0,1)$.

We also check that coverage holds at a lower SNR.

In [6]:
k_true = int(np.argmin(np.abs(mc_grid - Mc_true)))

def coverage(snr, R=400):
    amp = snr / np.sqrt(inner(_h_unit, _h_unit))
    Hg  = amp * fb[None, :]**(-7/6) * np.exp(1j * phase_0pn(fb[None, :], mc_grid[:, None]))
    hhg = 4 * df * np.sum(np.abs(Hg)**2 / Sb, axis=1).real
    Am  = 4 * df * Hg / Sb; del Hg
    h_t = amp * fb**(-7/6) * np.exp(1j * phase_0pn(fb, Mc_true))
    noise = (np.sqrt(Sb / (4*df))[None, :]
             * (rng.standard_normal((R, fb.size)) + 1j * rng.standard_normal((R, fb.size))))
    D = h_t[None, :] + noise
    lnLm   = (Am @ np.conj(D).T).real - 0.5 * hhg[:, None]     # [K, R]
    lnLmax = lnLm.max(0)
    w_true = 2 * (lnLmax - lnLm[k_true, :])                    # ~ chi^2_1
    cov    = {cl: float(np.mean(w_true <= chi2.ppf(cl, 1))) for cl in (0.68, 0.90, 0.95)}
    p      = chi2.cdf(w_true, df=1)
    sig    = 1.0 / np.sqrt(fisher_analytic(Mc_true, amp))
    pulls  = (mc_grid[lnLm.argmax(0)] - Mc_true) / sig
    return cov, p, pulls

for snr in (12.0, 30.0):
    cov, _, pulls = coverage(snr)
    print(f"SNR={snr:>5}:  68%->{cov[0.68]:.3f}   90%->{cov[0.90]:.3f}   "
          f"95%->{cov[0.95]:.3f}   pull mean/std = {pulls.mean():+.2f}/{pulls.std():.2f}")

cov, p, pulls = coverage(30.0, R=800)
fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 4.2))
a0.plot(np.sort(p), np.linspace(0, 1, p.size), lw=2, label="empirical")
a0.plot([0, 1], [0, 1], "k--", label="ideal")
a0.set_xlabel("nominal confidence level"); a0.set_ylabel("empirical coverage")
a0.set_title("P–P: coverage of Wilks intervals (SNR=30)"); a0.legend()
zz = np.linspace(-4, 4, 200)
a1.hist(pulls, bins=32, density=True, alpha=0.6, color="C0", label="MLE pulls")
a1.plot(zz, norm.pdf(zz), "k--", label=r"$\mathcal{N}(0,1)$")
a1.set_xlabel(r"$(\hat{\mathcal{M}} - \mathcal{M}_\mathrm{true})/\sigma$")
a1.set_title("pull distribution (SNR=30)"); a1.legend()
fig.tight_layout(); plt.show()

SNR= 12.0:  68%->0.688   90%->0.892   95%->0.950   pull mean/std = +0.01/1.00


SNR= 30.0:  68%->0.680   90%->0.892   95%->0.950   pull mean/std = -0.00/0.98


## Step 5 — Bayesian cross‑check

Finally, a 1‑D Bayesian posterior with a flat prior, $p(\mathcal{M}\,|\,d)\propto L(\mathcal{M})$,
evaluated on the grid. The **Bernstein–von Mises** theorem says the frequentist confidence
interval, the Fisher Gaussian, and the Bayesian credible interval all coincide at high SNR —
which is exactly what we should see here.

In [7]:
post = np.exp(lnL - lnL.max())
post /= np.trapezoid(post, mc_grid)
cdf  = np.concatenate([[0.0], np.cumsum(0.5 * (post[1:] + post[:-1]) * np.diff(mc_grid))])

def credible_interval(cl):
    lo = np.interp((1 - cl) / 2, cdf, mc_grid)
    hi = np.interp(1 - (1 - cl) / 2, cdf, mc_grid)
    return lo, hi

mean = np.trapezoid(mc_grid * post, mc_grid)
std  = np.sqrt(np.trapezoid((mc_grid - mean)**2 * post, mc_grid))

lo_f, hi_f = wilks_interval(0.90)
lo_b, hi_b = credible_interval(0.90)
print("90% intervals:")
print(f"  frequentist (Wilks)    : [{lo_f:.6f}, {hi_f:.6f}]")
print(f"  Bayesian    (credible) : [{lo_b:.6f}, {hi_b:.6f}]")
print(f"  Fisher      (+-1.645s) : [{mc_hat - 1.645*sig_fisher:.6f}, {mc_hat + 1.645*sig_fisher:.6f}]")
print(f"posterior mean/std = {mean:.6f} / {std:.3e}   (Fisher sigma = {sig_fisher:.3e})")

fig, ax = plt.subplots(figsize=(8, 4.3))
ax.plot(mc_grid, post, color="C3", label="Bayesian posterior (flat prior)")
ax.plot(mc_grid, np.exp(-0.5*(mc_grid-mc_hat)**2/sig_fisher**2) / (sig_fisher*np.sqrt(2*np.pi)),
        "--", color="C0", label="Fisher Gaussian")
ax.axvspan(lo_f, hi_f, color="C2", alpha=0.15, label="Wilks 90%")
ax.axvline(Mc_true, color="crimson", lw=1.2, label="truth")
ax.set_xlim(Mc_true - 5*sig_fisher, Mc_true + 5*sig_fisher)
ax.set_xlabel(r"$\mathcal{M}$ [$M_\odot$]"); ax.set_ylabel("posterior density")
ax.legend(); ax.set_title("Step 5 — frequentist vs Bayesian (Bernstein–von Mises)"); plt.show()

90% intervals:
  frequentist (Wilks)    : [1.218998, 1.219056]
  Bayesian    (credible) : [1.218998, 1.219056]
  Fisher      (+-1.645s) : [1.219001, 1.219054]
posterior mean/std = 1.219027 / 1.763e-05   (Fisher sigma = 1.590e-05)


## Takeaways & extensions

* The **profile/likelihood‑ratio interval (Wilks)**, the **Fisher/Laplace Gaussian**, and the
  **Bayesian credible interval** agree at high SNR, and the frequentist intervals have
  **correct coverage** — no MCMC required.
* The autodiff Hessian gives the Fisher matrix *exactly*, which is what makes this scale to
  many parameters (where grids/MCMC do not).

Natural next steps in this same framework:

* **Nuisance parameters** — free the amplitude/phase (profile analytically ⇒ the statistic
  becomes the matched‑filter $\rho^2$) and the coalescence time (whose matched‑filter
  landscape is *multimodal* — where naive Wilks under‑covers).
* **Boundaries & non‑Gaussianity** — add aligned spins / mass ratio near their physical edges
  and watch Fisher ≠ profile ≠ Bayesian (Chernoff boundary corrections).
* **PSD uncertainty** — replace the fixed‑PSD Gaussian likelihood with the Student‑t that
  results from a finite‑segment (Welch) PSD estimate, and re‑run the coverage test.